# Aditya Singh Chauhan — DATAGUARD_AI

This notebook contains runnable examples that exercise the core functionality of the DataGuard AI project included in this repository. It: 

- Imports the project modules from the repository
- Runs the sample data generators (optional)
- Runs the `core.profiler.profile` function on sample datasets and prints summaries
- Saves profiling outputs to JSON/CSV for submission

In [ ]:
# Cell 1: Ensure repository root is on sys.path so we can import project modules
import sys
from pathlib import Path
REPO_ROOT = Path('..').resolve() if (Path('.').resolve().name == 'submission_YourName_Project') else Path('.').resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('Repo root:', REPO_ROOT)

In [ ]:
# Cell 2: Imports from the project
import json
import pandas as pd
from dataclasses import asdict

# Import the profiler from the project
from core import profiler

print('Imports OK — profiler.profile is available:', callable(profiler.profile))

In [ ]:
# Cell 3: Optional — run the sample data generators to (re)create sample_data files.
# Uncomment the block below if you want to regenerate datasets in sample_data/
# from generate_sample_data import make_transactions, make_hr, make_hospital, make_ecommerce
# make_transactions()
# make_hr()
# make_hospital()
# make_ecommerce()
print('Skip generators by default to avoid long runs in notebook')

In [ ]:
# Cell 4: Helper to run the profiler on a CSV and return a dict-friendly result
from typing import Any, Dict

DATA_DIR = Path('sample_data')

def profile_csv(path: Path) -> Dict[str, Any]:
    df = pd.read_csv(path)
    result = profiler.profile(df)
    # Convert dataclasses to dicts for JSON-serialisable output
    out = asdict(result)
    # Columns is a list of ColumnProfile dataclasses — convert each
    out['columns'] = [asdict(c) for c in result.columns]
    return out

print('Helper defined')

In [ ]:
# Cell 5: Profile the new customer_profiles_enriched.csv and show summary scores
pfile = DATA_DIR / 'customer_profiles_enriched.csv'
if pfile.exists():
    prof = profile_csv(pfile)
    print('Profile for', pfile)
    print('Rows:', prof['total_rows'], 'Cols:', prof['total_cols'])
    print('Completeness score:', prof['completeness_score'])
    print('Consistency score:', prof['consistency_score'])
    print('Uniqueness score:', prof['uniqueness_score'])
    print('Validity score:', prof['validity_score'])
    # show top 5 columns by outlier_pct
    cols_sorted = sorted(prof['columns'], key=lambda c: c.get('outlier_pct',0), reverse=True)
    print('
Top columns by outlier_pct:')
    for c in cols_sorted[:5]:
        print('-', c['name'], 'outlier_pct=', c.get('outlier_pct'))
    # save JSON for submission
    (Path('.') / 'profile_customer_profiles_enriched.json').write_text(json.dumps(prof, indent=2))
    print('Saved profile_customer_profiles_enriched.json')
else:
    print('File not found:', pfile)

In [ ]:
# Cell 6: Profile the transaction dataset for a complementary view
tfile = DATA_DIR / 'customer_transactions.csv'
if tfile.exists():
    prof_t = profile_csv(tfile)
    print('Transaction dataset — rows:', prof_t['total_rows'], 'duplicates:', prof_t['duplicate_count'])
    # show a few schema issues if present
    if prof_t.get('schema_issues'):
        print('Schema issues found:')
        for s in prof_t['schema_issues']:
            print('-', s)
    (Path('.') / 'profile_customer_transactions.json').write_text(json.dumps(prof_t, indent=2))
    print('Saved profile_customer_transactions.json')
else:
    print('File not found:', tfile)

## Notes and next steps

- The notebook calls `core.profiler.profile` to produce structured profiling results.
- Use the generated JSON files as part of your submission (`profile_customer_profiles_enriched.json` and `profile_customer_transactions.json`).
- To run the full dashboard locally, follow the repository README: `streamlit run ui/app.py`.